# EZStats AI Worker — run on a Colab GPU

**The point of this notebook:** use Colab's GPU for the heavy steps while the code
stays on your PC, and get the results back automatically — *without re-uploading
the repo every time you change a line*.

## How it avoids re-uploading

| What | How it gets to Colab | When |
|---|---|---|
| **Code** (`src/`, run scripts) | `git pull` from GitHub | every run — seconds, incremental |
| **Models** (`artifacts/`, 315 MB) | Google Drive | **once** |
| **Videos** (`data/raw/`) | Google Drive | once per new video |
| **Results** (`outputs/<run>/`) | written to Drive -> Drive for Desktop syncs them to your PC | every run |

So the edit loop is: **edit locally -> `git push` -> re-run cell 4 -> run.**
No zipping, no manual upload, no `Compress-Archive` backslash problems.

---

## One-time setup (do this once, then never again)

1. **Install Google Drive for Desktop** on Windows and sign in. This is what makes
   results appear on your PC automatically.
2. In Drive, create a folder **`ezstats`** with two subfolders:
   - `ezstats/artifacts/` — copy your local `artifacts/` into it (315 MB: the
     player, ball and pitch models)
   - `ezstats/data/raw/`  — copy the match videos you want to process
3. Runtime -> Change runtime type -> **GPU (T4 is fine)** -> Save.
4. Run the cells below in order.

> Colab is Linux, so `run_pipeline_v2.ps1` will not run here. `run_pipeline_v2.py`
> is the portable runner with identical steps and flags — that is what this
> notebook calls, so local and Colab runs stay in sync.

### 1. Confirm we actually got a GPU
If this prints `No GPU`, fix the runtime type before continuing — everything below will still work, just at CPU speed.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or "No GPU")
import torch; print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

### 2. Mount Drive
Authorise when prompted. This is where the models, videos and results live.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/ezstats')
assert DRIVE.exists(), f"Create {DRIVE} in Drive first (see the setup steps above)."
print("Drive OK:", DRIVE)
for p in sorted(DRIVE.iterdir()):
    print("  ", p.name)

### 3. Get the code — clone the first time, pull every time after

This is the cell that replaces re-uploading. After you `git push` from your PC,
just re-run this cell and Colab has your latest code in a couple of seconds.

If the repo is private, generate a GitHub personal access token and use
`https://<TOKEN>@github.com/MattyKKS/EZStatsAIWorker.git` as the URL.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/MattyKKS/EZStatsAIWorker.git"
BRANCH   = "test"            # change to the branch you are working on
REPO     = Path('/content/EZStatsAIWorker')

if REPO.exists():
    print(subprocess.run(["git", "fetch", "--all"], cwd=REPO, text=True, capture_output=True).stdout)
    print(subprocess.run(["git", "checkout", BRANCH], cwd=REPO, text=True, capture_output=True).stdout)
    print(subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO, text=True, capture_output=True).stdout)
else:
    print(subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO)], text=True, capture_output=True).stdout)

print(subprocess.run(["git", "log", "--oneline", "-3"], cwd=REPO, text=True, capture_output=True).stdout)

### 4. Install dependencies

~2-3 minutes on a fresh runtime. Colab already ships torch with CUDA, so we do not
touch it. `sports` is Roboflow's helper package (pitch config, ByteTrack helpers)
and is not on PyPI, hence the git install.

> The last line **verifies the editable install actually worked**. `%pip` failures
> do not stop a Colab cell, so without this check a failed install shows up three
> cells later as a confusing `ModuleNotFoundError`.

In [ ]:
%pip install -q ultralytics supervision umap-learn transformers sentencepiece
%pip install -q git+https://github.com/roboflow/sports.git
%pip install -q -e /content/EZStatsAIWorker

# Verify — do NOT trust the cell finishing as proof the install succeeded.
import sys, importlib
print("python:", sys.version.split()[0])
for mod in ("ez_worker", "ultralytics", "supervision", "sports", "transformers"):
    try:
        importlib.import_module(mod)
        print(f"  OK   {mod}")
    except Exception as exc:
        raise SystemExit(
            f"  FAIL {mod}: {exc}\n\n"
            "If this is ez_worker, the editable install failed — scroll up for the pip error. "
            "A 'requires a different Python' error means pyproject.toml needs its "
            "requires-python widened for this Colab image; tell Claude the version above."
        )
try:
    import umap  # noqa: F401
    print("  OK   umap (dimensionality reduction active)")
except Exception:
    print("  NOTE umap unavailable — team clustering falls back to raw SigLIP features. "
          "Not fatal, but results may differ slightly from a local run.")

### 5. Link models and videos from Drive

Symlinks, not copies — no waiting for 315 MB to duplicate. `artifacts/` and
`data/` are gitignored, which is exactly why they come from Drive instead of git.

In [ ]:
import os
from pathlib import Path

REPO  = Path('/content/EZStatsAIWorker')
DRIVE = Path('/content/drive/MyDrive/ezstats')

for name in ("artifacts", "data"):
    link, target = REPO / name, DRIVE / name
    if not target.exists():
        raise SystemExit(f"Missing {target} in Drive — upload it (see setup step 2).")
    if link.is_symlink() or link.exists():
        if link.is_symlink():
            link.unlink()
        else:
            raise SystemExit(f"{link} exists and is not a symlink — remove it first.")
    os.symlink(target, link)
    print(f"{name} -> {target}")

print("\nVideos available:")
for v in sorted((DRIVE / 'data' / 'raw').glob('*.mp4')):
    print(f"  {v.name}  ({v.stat().st_size/1e6:.0f} MB)")

### 6. Run the videos

Set `RUNS` to whatever you want processed and the loop handles them **one after
another automatically** — no babysitting between clips.

> ### Keep the video ON
> `stats_video.mp4` is how you actually **check the events with your own eyes** —
> whether that "pass" was a pass, whether the goal got detected. A report full of
> numbers you cannot verify is worth very little. On CPU that render is ~80% of the
> runtime, which is exactly why it is worth being on a GPU. **You render here.**
> `--skip-video` exists only for fast local threshold tuning.

**Run `08fd33_4` first** — it is the fastest (750 frames) and it is the regression
gate. It should produce roughly 8 sensible events. If it does not, stop and say so
before trusting the other two.

Outputs go to Colab's local disk first; Drive is slow for the many small files a
run produces (player crops especially), so cell 7 copies the finished folders over.

In [ ]:
# (video path, source fps) — processed top to bottom, automatically.
RUNS = [
    ("data/raw/08fd33_4.mp4",          25),   # regression gate — run first
    ("data/raw/leo_messi_30pass.mp4",  60),   # priority: pink kit, real goal, worst camera drift
    ("data/raw/BrightonGoal.mp4",      25),   # noisiest clip — expect the weakest result
]
SKIP_VIDEO = False        # leave False: you need the video to verify events
GOAL_FRAMES = {}          # optional, e.g. {"data/raw/leo_messi_30pass.mp4": 6200}

import subprocess, sys, time
from pathlib import Path
REPO = Path('/content/EZStatsAIWorker')

run_dirs = {}
for video, fps in RUNS:
    print("\n" + "=" * 70)
    print(f"RUNNING  {video}   (fps={fps})")
    print("=" * 70, flush=True)
    t0 = time.time()

    cmd = [sys.executable, "run_pipeline_v2.py", "--video", video, "--source-fps", str(fps)]
    if SKIP_VIDEO:
        cmd.append("--skip-video")
    if video in GOAL_FRAMES:
        cmd += ["--goal-frame", str(GOAL_FRAMES[video])]

    rd = None
    proc = subprocess.Popen(cmd, cwd=REPO, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="")
        if line.startswith("Run dir:"):
            rd = line.split(":", 1)[1].strip()
    proc.wait()

    mins = (time.time() - t0) / 60
    if proc.returncode == 0 and rd:
        run_dirs[video] = rd
        print(f"\n>>> {video} OK in {mins:.1f} min -> {rd}")
    else:
        # keep going: one bad clip should not cost you the whole queue
        print(f"\n>>> {video} FAILED after {mins:.1f} min (exit {proc.returncode}) — continuing")

print("\n" + "=" * 70)
print("ALL RUNS FINISHED")
for v, d in run_dirs.items():
    print(f"  {v:<40} -> {d}")
if len(run_dirs) < len(RUNS):
    print(f"  {len(RUNS) - len(run_dirs)} run(s) failed — see the log above.")

### 7. Copy the results back

Writes every run folder to `ezstats/outputs/` in Drive. With Drive for Desktop
running, they appear on your PC by themselves — no download step.

The big intermediate `processed_video.mp4` is skipped (redundant with
`stats_video.mp4`, and the backend never serves it).

In [ ]:
import shutil
from pathlib import Path

REPO  = Path('/content/EZStatsAIWorker')
DRIVE = Path('/content/drive/MyDrive/ezstats')

for video, rd in run_dirs.items():
    src = REPO / rd
    if not src.exists():
        print(f"SKIP {video}: {src} missing")
        continue
    dst = DRIVE / 'outputs' / src.name
    dst.mkdir(parents=True, exist_ok=True)
    print(f"\n{video}  ->  {dst}")
    for item in sorted(src.iterdir()):
        if item.name == "processed_video.mp4":
            continue
        target = dst / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            shutil.copy2(item, target)
        print(f"   {item.name}")

print("\nDrive for Desktop will sync these to your PC under  G:\\My Drive\\ezstats\\outputs\\")

### 8. Summary of every run

Sanity-check the numbers before going back to your PC. What to look for:

- **`08fd33_4`** — roughly 8 events, 2 teams at about 13/13, possession NOT 50.0/50.0
  (that old value was an artefact of counting two touch events).
- **Messi** — a goal or a shot near the end, and passes attributed to *low, stable*
  track IDs rather than IDs in the hundreds.
- **Player counts** far above ~30 mean junk tracks are surviving (crowd, bench) —
  that is the known open item the pitch mask is meant to fix.

In [ ]:
import json
from collections import Counter
from pathlib import Path

REPO = Path('/content/EZStatsAIWorker')
for video, rd in run_dirs.items():
    rep = REPO / rd / 'match_report_merged.json'
    if not rep.exists():
        rep = rep.with_name('match_report.json')
    if not rep.exists():
        print(f"{video}: no report found"); continue
    d = json.loads(rep.read_text())
    print("=" * 70)
    print(f"{video}   [{rep.name}]   run={rd}")
    print("  players   :", len(d['players']), dict(Counter(p.get('team_id') for p in d['players'])))
    print("  possession:", d.get('possession'))
    print("  events    :", len(d['events']), dict(Counter(e['type'] for e in d['events'])))
    for e in d['events']:
        print(f"    {e['type']:<13} t={e['time_s']:<8} {e.get('actor_label')} -> {e.get('target_label')}")

---
## The loop from here

1. Edit code on your PC.
2. `git add -A && git commit -m "..." && git push`
3. In Colab: re-run **cell 3** (`git pull`) and **cell 6** (run).
4. Results land in Drive and sync to your PC.

## Honest caveats

- **Colab sessions are ephemeral.** `/content` is wiped when the runtime recycles.
  Anything you care about must go to Drive (cell 7). Re-running cells 3-5 rebuilds
  the environment in ~3 minutes.
- **Idle disconnects.** Colab drops idle sessions; keep the tab open for long runs.
- **Drive sync is not instant** for large files — a 260 MB `stats_video.mp4` takes
  a while to appear on the PC.
- **Uncommitted local changes do not reach Colab.** The `git reset --hard` in cell 3
  is deliberate — it guarantees Colab matches your pushed branch exactly — but it
  means anything you forgot to push is simply not there.
- **`processed_video.mp4` is skipped** on copy-back by design.